Deep learning jounrey begins ...

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

class Perceptron:
    def __init__(self, learning_rate=0.01, n_iterations=1000):
        self.lr = learning_rate
        self.n_iters = n_iterations
        self.weights = None
        self.bias = None

    def _activation(self, x):
        return np.where(x >= 0, 1, 0) #if x>=0 then 1 else 0 (binary step function)

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0

        for epoch in range(self.n_iters):
            for idx, x_i in enumerate(X):
                linear_output = np.dot(x_i, self.weights) + self.bias
                y_predicted = self._activation(linear_output)

                # Perceptron update rule
                update = self.lr * (y[idx] - y_predicted) #loss (y-y_hat)
                self.weights += update * x_i ##adjust weights
                self.bias += update ##adjust bias

    def predict(self, X):
        linear_output = np.dot(X, self.weights) + self.bias
        return self._activation(linear_output)


# --- Generate a simple binary classification dataset ---
X, y = make_classification(
    n_samples=500,
    n_features=2,        # 2 features so it's easy to reason about
    n_informative=2,
    n_redundant=0,
    n_clusters_per_class=1,
    random_state=42
)

# --- Split into train and test ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# --- Train the perceptron ---
p = Perceptron(learning_rate=0.01, n_iterations=1000)
p.fit(X_train, y_train)

# --- Predict and evaluate ---
predictions = p.predict(X_test)
acc = accuracy_score(y_test, predictions)

print("=== Perceptron Results ===")
print(f"Weights:       {p.weights}")
print(f"Bias:          {p.bias:.4f}")
print(f"Predictions:   {predictions[:10]}  ← first 10")
print(f"Actual labels: {y_test[:10]}        ← first 10")
print(f"Accuracy:      {acc * 100:.2f}%")

=== Perceptron Results ===
Weights:       [-0.00620923  0.01812331]
Bias:          0.0000
Predictions:   [1 1 0 1 0 1 0 1 1 1]  ← first 10
Actual labels: [1 1 0 1 0 1 0 1 0 1]        ← first 10
Accuracy:      91.00%


However we must note that this preceptron will simply fail on data that cannot be linearly seperated hence we use a network or such neurons - neural net - to learn about more complex data patterns in real life.

---
## PyTorch Implementation of a Perception (notice how smaller it becomes)


In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# ── 1. Single Perceptron in PyTorch ─────────────────────────────
class PyTorchPerceptron(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)  # z = w·x + b
        self.sigmoid = nn.Sigmoid()            # y_hat = σ(z)

    def forward(self, x):
        z = self.linear(x)
        y_hat = self.sigmoid(z)
        return z, y_hat

# Convert data to PyTorch Tensors
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
model = PyTorchPerceptron(input_dim=2)

with torch.no_grad():
    z_sample, y_hat_sample = model(X_test_tensor[:5])

print("=== PyTorch Forward Pass Sample (First 5) ===")
print("Linear output (z):", z_sample.squeeze().numpy())
print("Sigmoid output (y_hat):", y_hat_sample.squeeze().numpy())

# ── 2. Visualizing Activation Functions in PyTorch ──────────────
x_range = torch.linspace(-5, 5, 200)
activations = {
    "Sigmoid": nn.Sigmoid(),
    "Tanh": nn.Tanh(),
    "ReLU": nn.ReLU(),
    "LeakyReLU": nn.LeakyReLU(negative_slope=0.1)
}

plt.figure(figsize=(10, 5))
for name, func in activations.items():
    y = func(x_range)
    plt.plot(x_range.numpy(), y.numpy(), label=name, linewidth=2)

plt.axhline(0, color="gray", linestyle="--", alpha=0.6)
plt.axvline(0, color="gray", linestyle="--", alpha=0.6)
plt.title("Activation Functions in PyTorch")
plt.xlabel("Linear Output (z)")
plt.ylabel("Activated Output (y_hat)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()
